# Person 4 (Neha): sample candidates for the third set, Coding cells
Owner: Person 4 (N). Output: 25 Easy (MBPP) and 25 Hard (LeetCode) candidate rows, no overlap with the 232 existing pairs.
Everything here is **exploratory data-building**: the filters below are written BEFORE any router score exists. Do not change them after scoring.

In [ ]:
!pip install -q datasets openpyxl

## 1. Upload the two files that list what is already used
Upload `confirmatory_v1.csv` (from `data/processed/` in the repo) and `benchmark_v2.xlsx` (from `data/raw/`).

In [ ]:
from google.colab import files
up = files.upload()   # select confirmatory_v1.csv AND benchmark_v2.xlsx

In [ ]:
import pandas as pd, re

conf  = pd.read_csv('confirmatory_v1.csv')
pilot = pd.read_excel('benchmark_v2.xlsx', sheet_name='Benchmark')

used_ids   = set(conf['source_id'].astype(str))                 # e.g. MBPP-605, LeetCode-33
used_texts = list(conf['english']) + list(pilot['english'])      # for the near-duplicate check
print(len(conf), 'confirmatory rows,', len(pilot), 'pilot rows,', len(used_ids), 'used source ids')

## 2. Easy cell: MBPP
MBPP has task_ids 1 to 974. We load every split together and skip anything already used.

In [ ]:
from datasets import load_dataset, concatenate_datasets

mbpp = load_dataset('google-research-datasets/mbpp', 'full')   # splits: train, test, validation, prompt
m = pd.concat([mbpp[s].to_pandas() for s in mbpp.keys()], ignore_index=True)
print(m.shape, m.columns.tolist())      # expect: task_id, text, code, test_list, ...
print(m['task_id'].min(), m['task_id'].max())

In [ ]:
m['source_id'] = 'MBPP-' + m['task_id'].astype(str)

def words(t): return set(re.findall(r'[a-z0-9_]+', t.lower()))
def too_similar(t, pool, thr=0.6):
    w = words(t)
    return any(len(w & words(p)) / max(1, len(w | words(p))) >= thr for p in pool)

# FILTERS (fixed before scoring):
#  1) not already used  2) 40 to 200 characters  3) no mention of images/figures
#  4) not a near-duplicate (word overlap >= 0.6) of any existing English text
m['len'] = m['text'].str.len()
ok = (~m['source_id'].isin(used_ids)) & m['len'].between(40, 200) \
     & ~m['text'].str.contains(r'image|figure|diagram|picture', case=False, regex=True)
cand = m[ok].copy()
cand = cand[~cand['text'].apply(lambda t: too_similar(t, used_texts))]
print('candidates after filters:', len(cand), 'of', len(m))

easy = cand.sample(40, random_state=0)      # 40 drawn, you trim to 25 by hand and LOG each drop
easy_out = pd.DataFrame({'difficulty': 'Easy', 'source': 'MBPP',
                         'source_id': easy['source_id'].values,
                         'mbpp_original_text': easy['text'].values})
easy_out.head(10)

## 3. Hard cell: LeetCode
LeetCode has no clean public dataset we can verify, and its text must not be released. So, like Aryan did, we pick problem **numbers** and write the English **in our own words**.
Rubric (agreed in the plan): *an algorithm or design problem with constraints.*
**Before using a number, open the problem on leetcode.com and check that the number matches the title below.** I wrote the titles from memory, so verify them.

In [ ]:
leet = {
 3:'Longest Substring Without Repeating Characters', 10:'Regular Expression Matching', 15:'3Sum',
 22:'Generate Parentheses', 32:'Longest Valid Parentheses', 39:'Combination Sum', 45:'Jump Game II',
 49:'Group Anagrams', 55:'Jump Game', 56:'Merge Intervals', 62:'Unique Paths', 79:'Word Search',
 91:'Decode Ways', 98:'Validate Binary Search Tree', 128:'Longest Consecutive Sequence', 133:'Clone Graph',
 139:'Word Break', 152:'Maximum Product Subarray', 198:'House Robber', 207:'Course Schedule',
 208:'Implement Trie', 215:'Kth Largest Element in an Array', 238:'Product of Array Except Self',
 239:'Sliding Window Maximum', 295:'Find Median from Data Stream', 297:'Serialize and Deserialize Binary Tree',
 300:'Longest Increasing Subsequence', 322:'Coin Change', 329:'Longest Increasing Path in a Matrix',
 347:'Top K Frequent Elements', 399:'Evaluate Division', 410:'Split Array Largest Sum',
 416:'Partition Equal Subset Sum', 494:'Target Sum', 560:'Subarray Sum Equals K', 621:'Task Scheduler',
 695:'Max Area of Island', 739:'Daily Temperatures',
}
pool = pd.DataFrame({'num': list(leet), 'title': list(leet.values())})
pool['source_id'] = 'LeetCode-' + pool['num'].astype(str)
pool = pool[~pool['source_id'].isin(used_ids)]
print(len(pool), 'LeetCode problems left after removing used ones')

hard = pool.sample(25, random_state=0)       # exactly 25, seed 0
hard_out = pd.DataFrame({'difficulty': 'Hard', 'source': 'LeetCode',
                         'source_id': hard['source_id'].values,
                         'mbpp_original_text': 'WRITE YOUR OWN WORDING. Problem: ' + hard['title'].values})
hard_out.head(10)

## 4. Build the two output files
`coding_sheet_N.csv` has the exact column names the scoring code reads. `coding_reference_N.csv` is only a helper for you. Do **not** merge it into the master file.

In [ ]:
both = pd.concat([easy_out, hard_out], ignore_index=True)

sheet = pd.DataFrame({
    'pair_id': '',                      # Aryan assigns the ID block
    'task_type': 'Coding',
    'difficulty': both['difficulty'],
    'source': both['source'],
    'source_id': both['source_id'],
    'written_by': 'N',
    'checked_by': 'A',                  # Aryan is your second reader
    'check_status': '',                 # reviewer fills OK or FLAGGED
    'check_note': '',
    'english': '',                      # you write these
    'hinglish': '',
})
sheet.to_csv('coding_sheet_N.csv', index=False, encoding='utf-8')
both[['source_id', 'difficulty', 'mbpp_original_text']].to_csv('coding_reference_N.csv', index=False, encoding='utf-8')
print(sheet.groupby(['difficulty', 'source']).size())

from google.colab import files
files.download('coding_sheet_N.csv'); files.download('coding_reference_N.csv')